# Customer Churn — Data Quality & Exploratory Analysis
**Task 1 | Beginner-friendly reproducible notebook**

Business question: What does this customer sample tell us about data quality and initial churn patterns?

> The dataset has only 15 synthetic records. Treat patterns as exploratory, not causal or statistically conclusive.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('customer_churn_sample.csv')
print('Shape:', df.shape)
display(df.head())


## 1. Profile columns, missing values and duplicates


In [ ]:
profile = pd.DataFrame({
    'data_type': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2),
    'unique_values': df.nunique(dropna=True)
})
display(profile)
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate CustomerIDs:', df['CustomerID'].duplicated().sum())
display(df.describe(include='all').T)


## 2. Outlier screening using the 1.5 × IQR rule
Outlier flags are prompts for review, not automatic grounds for deleting records.


In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
outlier_rows = []
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = df.loc[(df[col] < lower) | (df[col] > upper), col].tolist()
    outlier_rows.append({'column': col, 'lower_bound': lower, 'upper_bound': upper, 'flagged_values': flagged})
display(pd.DataFrame(outlier_rows))


## 3. Cleaning decisions
- Trim leading/trailing whitespace in text fields.
- Do not impute: no missing values were found.
- Do not remove rows: no exact duplicate rows or duplicate customer IDs were found.
- Do not cap/delete IQR outliers: none were flagged.
- Preserve supplied categories and column names.
- Treat CustomerID as an identifier, not a numeric predictor.


In [ ]:
clean = df.copy()
for col in clean.select_dtypes(include='object').columns:
    clean[col] = clean[col].astype('string').str.strip()

# Validation checks
assert clean.isna().sum().sum() == 0
assert clean.duplicated().sum() == 0
assert clean['CustomerID'].is_unique
assert set(clean['Churn'].dropna().str.lower()).issubset({'yes', 'no'})
assert (clean['Age'] > 0).all()
assert (clean['TenureMonths'] >= 0).all()
assert (clean['MonthlyCharges'] >= 0).all()
assert (clean['TotalCharges'] >= 0).all()
assert (clean['SupportTickets'] >= 0).all()

charge_diff = (clean['TotalCharges'] - clean['TenureMonths'] * clean['MonthlyCharges']).abs()
print('Rows failing TotalCharges arithmetic check:', int((charge_diff > 0.01).sum()))
clean.to_csv('cleaned_customer_churn.csv', index=False)
print('Saved cleaned_customer_churn.csv')


## 4. Summary statistics and churn KPIs


In [ ]:
display(clean.select_dtypes(include='number').describe().T)
churned = (clean['Churn'].str.lower() == 'yes').sum()
retained = (clean['Churn'].str.lower() == 'no').sum()
print(f'Customers: {len(clean)}')
print(f'Churn: {churned} ({churned / len(clean):.2%})')
print(f'Retention: {retained} ({retained / len(clean):.2%})')
display(clean['Churn'].value_counts())


## 5. Exploratory charts


In [ ]:
clean['Churn'].value_counts().reindex(['Yes', 'No']).plot(kind='bar')
plt.title('Customer Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Number of customers')
plt.tight_layout()
plt.show()

contract_churn = clean.groupby('ContractType')['Churn'].apply(lambda s: (s.str.lower() == 'yes').mean() * 100)
display(contract_churn.sort_values(ascending=False))
contract_churn.sort_values().plot(kind='barh')
plt.title('Churn Rate by Contract Type')
plt.xlabel('Churn rate (%)')
plt.tight_layout()
plt.show()

display(clean.groupby('Churn')['SupportTickets'].mean())
clean.groupby('Churn')['SupportTickets'].mean().reindex(['Yes', 'No']).plot(kind='bar')
plt.title('Average Support Tickets by Churn Status')
plt.xlabel('Churn')
plt.ylabel('Average support tickets')
plt.tight_layout()
plt.show()


## 6. Findings and limitations
- The sample contains 15 records and 11 columns.
- No missing values, exact duplicate rows, duplicate CustomerIDs or IQR-flagged numeric outliers were found.
- Sample churn is 46.67% (7/15); retention is 53.33% (8/15).
- All month-to-month customers in this sample churned; all one-year and two-year contract customers stayed.
- Churned customers have a higher average support-ticket count in this sample.
- TotalCharges matches TenureMonths × MonthlyCharges for every record.

**Limitations:** With only 15 records, segment rates are unstable. These patterns do not establish causation or predict future churn. Currency is not specified in the source.
